In [ ]:
from pathlib import Path
import os
DATA_DIR = Path.cwd() / 'Data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'Data'

In [ ]:
import pandas as pd

# Load medications
meds = pd.read_csv(DATA_DIR / 'medications.csv')

# Standardize column names
meds.columns = meds.columns.str.lower()

# Filter to relevant columns
meds_filtered = meds[["patient", "description", "start", "stop", "totalcost"]]

meds_filtered

#### problem 1

In [ ]:
# Rename columns
# meds_filtered.rename(columns={
#     "patient": "patient_id",
#     "description": "medication_name",
#     "start": "start_date",
#     "stop": "end_date",
#     "totalcost": "total_cost"
# })

meds_filtered.rename(columns={
    "patient": "patient_id",
    "description": "medication_name",
    "start": "start_date",
    "stop": "end_date",
    "totalcost": "total_cost"
}, inplace= True)

meds_filtered

#### problem 2

In [ ]:
# Cast date columns
meds_filtered["start_date"] = pd.to_datetime(meds_filtered["start_date"])
meds_filtered["end_date"] = pd.to_datetime(meds_filtered["end_date"])

# Calculate duration in days
# meds_filtered["duration_days"] = (
#     meds_filtered["start_date"] - meds_filtered["end_date"]
# ).dt.days

meds_filtered["duration_days"] = (
    meds_filtered["end_date"] - meds_filtered["start_date"]
).dt.days

meds_filtered

#### problem 3

In [ ]:
# Deduplicate on patient and medication
meds_deduped = meds_filtered.drop_duplicates(["patient_id", "medication_name"])

# Filter to active medications only (no end date = still active)
# meds_active = meds_deduped[meds_deduped["end_date"] != None]
meds_active = meds_deduped[
    meds_deduped["end_date"].isna()
]
meds_active

#### problem 4

In [ ]:
# Compute average cost per medication
# avg_cost = meds_deduped.groupby("medication_name").agg(
#     avg_cost=("total_cost", "mean")
# ).sort_values("avg_cost", ascending=True)

avg_cost = meds_active.groupby("medication_name").agg(
    avg_cost=("total_cost", "mean")
).sort_values("avg_cost", ascending=True)
avg_cost


#### problem 5

In [ ]:
# Write output
os.makedirs("output", exist_ok=True) 

# Write output
avg_cost.to_csv(
    "output/avg_med_cost.csv",
    index=True
)